# 🌳 Refauna SLM — Qwen3.5-2B com fine-tuning (LoRA 16-bit) + RAG

Segunda rodada do experimento da palestra *Pequenos modelos, conhecimento especializado*: **o mesmo pipeline do Gemma 4 E2B, trocando só o modelo-base.**

| O que é igual ao notebook do Gemma | O que muda |
|---|---|
| Os **mesmos** trechos, índice de busca e exemplos de treino/validação/teste (lidos do Drive, nada é regerado nem cobrado de novo) | Modelo-base: `unsloth/Qwen3.5-2B` (Apache 2.0, 2,3 B parâmetros, arquitetura híbrida Gated DeltaNet + atenção) |
| O **mesmo** system prompt e formato `CONTEXTO: [1] ... PERGUNTA:` | **LoRA 16-bit em vez de QLoRA**: a Unsloth não recomenda treino 4-bit no Qwen3.5 (o 2B usa só ~5 GB em bf16) |
| A **mesma** avaliação A/B/C/D com o mesmo juiz | Template de chat ChatML (`<|im_start|>`) em **modo sem raciocínio** (`enable_thinking=False`) |
| Os mesmos hiperparâmetros (r=16, 2 épocas, lr 2e-4) | Exige **transformers v5** (a instalação abaixo cuida disso) |

**Pré-requisito:** ter rodado o notebook do Gemma até a seção 5 — este notebook lê de `refauna-slm/data/`:
`chunks.jsonl`, `chunk_embeddings.npy`, `docs.jsonl` e `sft/{train,val,test}.jsonl`.

Os mesmos Secrets do Colab (`HF_TOKEN`, `ANTHROPIC_API_KEY`) servem aqui. GPU: L4 ou A100.

## 0. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Instalação oficial da Unsloth para o Qwen3.5 (fixa transformers v5, obrigatório para este modelo)
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
# Bibliotecas do RAG/avaliação ANTES de fixar o transformers (para não serem rebaixadas depois)
!uv pip install -qqq chromadb rank-bm25 "sentence-transformers>=5.1" anthropic
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

import importlib.metadata as im
faltando = [m for m in ["unsloth", "chromadb", "rank_bm25", "sentence_transformers", "anthropic"] if importlib.util.find_spec(m) is None]
print("transformers", im.version("transformers"), "| sentence-transformers", im.version("sentence-transformers"))
print("✅ todas as bibliotecas instaladas" if not faltando else f"❌ faltando: {faltando}")

> Se o Colab pedir para **reiniciar a sessão**, reinicie e continue a partir da célula abaixo (sem reinstalar). O `transformers` precisa aparecer como **5.x**.

In [ ]:
import unsloth  # importe a Unsloth antes de transformers/trl
import os, re, json, random, time, shutil, unicodedata, csv
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from google.colab import drive, userdata

drive.mount("/content/drive")

# ======================= CONFIGURAÇÃO =======================
HF_USERNAME  = "iamrodrigocoelho"
HF_PRIVATE   = True

PROJECT_DIR  = Path("/content/drive/MyDrive/refauna-slm")
if not (PROJECT_DIR / "training-files").exists():
    found = sorted(Path("/content/drive/MyDrive").glob("**/training-files"))
    assert found, "Pasta 'training-files' não encontrada no Meu Drive. Ajuste PROJECT_DIR."
    PROJECT_DIR = found[0].parent
DATA_DIR  = PROJECT_DIR / "data"
OUT_DIR   = PROJECT_DIR / "outputs"
GOLD_FILE = PROJECT_DIR / "gold_questions.jsonl"
LOCAL_DIR = Path("/content/work"); LOCAL_DIR.mkdir(parents=True, exist_ok=True)

for f in ["chunks.jsonl", "docs.jsonl", "sft/train.jsonl", "sft/val.jsonl", "sft/test.jsonl"]:
    assert (DATA_DIR / f).exists(), f"Falta {DATA_DIR/f}. Rode antes o notebook do Gemma até a seção 5."

BASE_MODEL   = "unsloth/Qwen3.5-2B"
EMBED_MODEL  = "google/embeddinggemma-300m"   # o MESMO do índice criado no notebook do Gemma
GEN_MODEL    = "claude-sonnet-5-5"            # juiz da avaliação (o mesmo usado no Gemma)

LOAD_IN_4BIT = False   # Qwen3.5: LoRA 16-bit (QLoRA não é recomendado pela Unsloth para esta família)
MAX_SEQ_LEN  = 4096
LORA_R       = 16
EPOCHS       = 2
LR           = 2e-4
TOP_K        = 4
SEED         = 3407
RUN_NAME     = "qwen3.5-2b-refauna-lora16"

REPO_LORA   = f"{HF_USERNAME}/qwen3.5-2b-refauna-lora"
REPO_MERGED = f"{HF_USERNAME}/qwen3.5-2b-refauna"
REPO_GGUF   = f"{HF_USERNAME}/qwen3.5-2b-refauna-GGUF"

random.seed(SEED)
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
from huggingface_hub import login
login(token=HF_TOKEN)
print("GPU:", torch.cuda.get_device_name(0), "| projeto:", PROJECT_DIR)

In [ ]:
def write_jsonl(path, rows):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

### Contrato de prompt — copiado do notebook do Gemma
Mesmo system prompt, mesmo formato de contexto, mesma frase de abstenção. É isso que torna a comparação justa.

In [ ]:
SYSTEM_PROMPT = (
    "Você é o Assistente Refauna, especialista em defaunação, refaunação, reintrodução de fauna, "
    "dispersão de sementes, interações ecológicas e conservação da Mata Atlântica. "
    "Responda sempre em português do Brasil, de forma clara, didática e precisa. "
    "Quando houver CONTEXTO, use somente as informações dele e cite os trechos usados no formato [n]. "
    "Se o CONTEXTO não contiver a resposta, diga explicitamente que não encontrou essa informação "
    "nos documentos do Refauna e não invente dados, números, datas ou nomes de espécies."
)

ABSTAIN_ANSWERS = [
    "Não encontrei essa informação nos documentos do Refauna fornecidos no contexto. Os trechos disponíveis tratam de outros aspectos e não permitem responder com segurança.",
    "Os trechos recuperados não contêm a resposta para essa pergunta. Prefiro não especular: não encontrei essa informação nos documentos do Refauna.",
    "Com base no contexto fornecido, não é possível responder. Não encontrei nos documentos do Refauna dados sobre esse ponto específico.",
    "Não encontrei essa informação nos documentos do Refauna. O contexto disponível aborda temas relacionados, mas não traz esse dado.",
    "Essa informação não aparece nos trechos dos documentos do Refauna que recebi, então não consigo responder sem correr o risco de inventar.",
]
ABSTAIN_RE = re.compile(r"não (encontrei|consigo responder|é possível responder|há informaç|aparece)|não contêm a resposta", re.I)

def is_abstention(text):
    return bool(ABSTAIN_RE.search(text or ""))

NAME_SUFFIX = {"jr", "júnior", "junior", "filho", "neto", "sobrinho"}

def short_ref(c):
    """'(Cid et al., 2014 — Título curto)'. Aceita autores separados por ';' (planilha) ou ','."""
    raw = (c.get("authors") or "").strip()
    has_etal = bool(re.search(r"\bet al\b", raw))
    raw = re.sub(r"\bet al\.?", "", raw).strip(" ;,")
    parts = [a.strip() for a in re.split(r";" if ";" in raw else r",| & | and ", raw) if a.strip()]
    first = parts[0].split(",")[0] if parts else ""          # "Fernandez, F." -> "Fernandez"
    words = [w for w in first.split() if w.rstrip(".").lower() not in NAME_SUFFIX]
    surname = words[-1] if words else ""
    year = c.get("year") or "s.d."
    title = (c.get("title") or c.get("doc_id", "")).strip()
    title = title if len(title) <= 60 else title[:57].rstrip() + "..."
    who = (f"{surname} et al., " if (len(parts) > 1 or has_etal) else f"{surname}, ") if surname else ""
    return f"({who}{year} — {title})"

def format_context(chunks):
    return "\n".join(f"[{i}] {short_ref(c)} {c['text'].strip()}" for i, c in enumerate(chunks, 1))

def build_user_message(question, chunks=None):
    if chunks:
        return f"CONTEXTO:\n{format_context(chunks)}\n\nPERGUNTA: {question.strip()}"
    return question.strip()

def build_messages(question, chunks=None):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_message(question, chunks)}]

PT_SW = set("de que não para com uma os as dos das em é são também pela pelo mais como sua seu ao".split())
EN_SW = set("the and of to is are with for that this which were from by their".split())
def is_ptbr(text):
    toks = re.findall(r"\w+", (text or "").lower())
    return sum(t in PT_SW for t in toks) >= sum(t in EN_SW for t in toks)

## 1. Dados e índice (reaproveitados)
Nada de ler PDFs nem gerar perguntas: carregamos os trechos, os embeddings já calculados e o split treino/validação/teste do Drive. O índice híbrido (EmbeddingGemma + BM25) é remontado em memória em segundos.

In [ ]:
docs   = read_jsonl(DATA_DIR / "docs.jsonl")
chunks = read_jsonl(DATA_DIR / "chunks.jsonl")
chunk_by_id = {c["chunk_id"]: c for c in chunks}
train, val, test = (read_jsonl(DATA_DIR / "sft" / f"{s}.jsonl") for s in ("train", "val", "test"))
print(f"{len(docs)} documentos | {len(chunks)} trechos | train={len(train)} val={len(val)} test={len(test)}")
print("Composição do treino:", dict(Counter(e["kind"] for e in train)))

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi

embedder = SentenceTransformer(EMBED_MODEL, device="cuda", model_kwargs={"torch_dtype": torch.float32})

def index_text(c):
    return f"{c['title']} — {c['section']}\n{c['text']}"

emb_path = DATA_DIR / "chunk_embeddings.npy"
doc_emb = np.load(emb_path) if emb_path.exists() else None
if doc_emb is None or len(doc_emb) != len(chunks):
    print("Embeddings salvos ausentes ou desatualizados — recalculando...")
    doc_emb = embedder.encode_document([index_text(c) for c in chunks], batch_size=32, normalize_embeddings=True)

chroma = chromadb.EphemeralClient()
col = chroma.create_collection("refauna", metadata={"hnsw:space": "cosine"})
for i in range(0, len(chunks), 500):
    b = chunks[i:i + 500]
    col.add(ids=[c["chunk_id"] for c in b], embeddings=doc_emb[i:i + 500].tolist(), documents=[c["text"] for c in b])
print("Índice denso:", col.count(), "vetores")

In [ ]:
STOP = set(("a o e de da do das dos em no na nos nas um uma para por com que se ao os as sao "
            "the of and to in is for on with by as are was were be this that from at an or").split())

def bm25_tokens(s):
    s = unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in STOP and len(t) > 1]

def rrf_fuse(rankings, rrf_k=60):
    fused = defaultdict(float)
    for ranking in rankings:
        for rank, cid in enumerate(ranking):
            fused[cid] += 1.0 / (rrf_k + rank + 1)
    return sorted(fused, key=fused.get, reverse=True)

def select_diverse(ranked_ids, by_id, k, exclude_ids=()):
    """Remove excluídos e chunks vizinhos (sobrepostos) do mesmo documento."""
    out = []
    for cid in ranked_ids:
        if cid in exclude_ids:
            continue
        c = by_id[cid]
        if any(o["doc_id"] == c["doc_id"] and abs(o["idx"] - c["idx"]) <= 1 for o in out):
            continue
        out.append(c)
        if len(out) == k:
            break
    return out

In [ ]:
bm25 = BM25Okapi([bm25_tokens(index_text(c)) for c in chunks])
chunk_ids = [c["chunk_id"] for c in chunks]

def hybrid_search(query, k=TOP_K, n_candidates=20, exclude_ids=()):
    q = embedder.encode_query(query, normalize_embeddings=True)
    dense = col.query(query_embeddings=[q.tolist()], n_results=min(n_candidates, len(chunks)))["ids"][0]
    scores = bm25.get_scores(bm25_tokens(query))
    lexical = [chunk_ids[i] for i in np.argsort(scores)[::-1][:n_candidates]]
    return select_diverse(rrf_fuse([dense, lexical]), chunk_by_id, k, exclude_ids)

for q in ["O que é refaunação e por que ela importa para a Mata Atlântica?",
          "Quais animais foram reintroduzidos no Parque Nacional da Tijuca?",
          "Qual o papel das cutias na dispersão de sementes?"]:
    print(f"\n❓ {q}")
    for i, c in enumerate(hybrid_search(q), 1):
        print(f"  [{i}] {short_ref(c)} · {c['section']} · p.{c['page_start']} :: {c['text'][:110]}...")

In [ ]:
def retrieval_metrics(items, k=TOP_K):
    hit = doc_hit = 0; rr = 0.0
    for e in items:
        src = chunk_by_id[e["source_chunk_ids"][0]]
        res = hybrid_search(e["question"], k=20)
        pos = next((i for i, c in enumerate(res) if c["doc_id"] == src["doc_id"] and abs(c["idx"] - src["idx"]) <= 1), None)
        hit += pos is not None and pos < k
        rr += 1 / (pos + 1) if pos is not None else 0
        doc_hit += any(c["doc_id"] == src["doc_id"] for c in res[:k])
    n = len(items)
    return {"n": n, f"recall@{k}": hit / n, "mrr@20": rr / n, f"doc_recall@{k}": doc_hit / n}

ret_items = [e for e in test if e["kind"] in ("rag", "unanswerable")]
retrieval_report = retrieval_metrics(ret_items)
print(retrieval_report)

## 2. Fine-tuning do Qwen3.5-2B com LoRA 16-bit

- O modelo-base é carregado em **bf16** (sem compressão) e congelado; só os adaptadores LoRA são treinados.
- LoRA nas projeções de atenção e MLP (`q/k/v/o`, `gate/up/down`), como na receita oficial da Unsloth para o Qwen3.5. Visão fica de fora.
- **Modo sem raciocínio:** o Qwen3.5 sabe "pensar" num bloco `<think>...</think>` antes de responder. Para um assistente de RAG curto e citável, treinamos e usamos com `enable_thinking=False` (o padrão do 2B).
- O primeiro passo de treino pode demorar alguns minutos: a Unsloth compila os kernels da camada Gated DeltaNet.

In [ ]:
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
try:
    from unsloth import FastLanguageModel as Loader
    model, tokenizer = Loader.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=False, load_in_16bit=True,
        full_finetuning=False, token=HF_TOKEN)
    model = Loader.get_peft_model(
        model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none", target_modules=TARGETS,
        use_gradient_checkpointing="unsloth", random_state=SEED, max_seq_length=MAX_SEQ_LEN)
except Exception as e:   # o Qwen3.5 é multimodal; se o carregador de texto recusar, usa o genérico
    print("FastLanguageModel falhou, usando FastModel:", type(e).__name__, e)
    from unsloth import FastModel as Loader
    model, tokenizer = Loader.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=False, full_finetuning=False, token=HF_TOKEN)
    model = Loader.get_peft_model(
        model, finetune_vision_layers=False, finetune_language_layers=True,
        finetune_attention_modules=True, finetune_mlp_modules=True,
        r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none", random_state=SEED)

tok = getattr(tokenizer, "tokenizer", tokenizer)   # tokenizer de texto (se vier um processor multimodal)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Carregador: {Loader.__name__} | treináveis: {trainable/1e6:.1f}M de {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

In [ ]:
USER_TAG, ASSIST_TAG = "<|im_start|>user\n", "<|im_start|>assistant\n"

def render(messages, generation=False):
    """Template de chat nativo do Qwen3.5, sempre em modo sem raciocínio."""
    for t in (tok, tokenizer):
        if getattr(t, "chat_template", None):
            return t.apply_chat_template(messages, tokenize=False, add_generation_prompt=generation, enable_thinking=False)
    raise RuntimeError("Tokenizer sem chat_template")

# Na inferência sem raciocínio, o template abre a resposta com um bloco de pensamento vazio.
# Garantimos que os textos de TREINO tenham exatamente a mesma abertura (treino == inferência).
gen_prompt = render([{"role": "user", "content": "Teste"}], generation=True)
THINK_PREFIX = gen_prompt.split(ASSIST_TAG)[-1]
print("Prompt de geração:", repr(gen_prompt[-120:]))
print("Abertura da resposta:", repr(THINK_PREFIX))

def to_training_text(messages):
    text = render(messages)
    cut = text.rfind(ASSIST_TAG) + len(ASSIST_TAG)
    if THINK_PREFIX and not text[cut:].startswith(THINK_PREFIX):
        text = text[:cut] + THINK_PREFIX + text[cut:].lstrip()
    return text

sample = to_training_text(train[0]["messages"])
assert USER_TAG in sample and ASSIST_TAG in sample, "Template inesperado: confira os marcadores ChatML"
print("\nComo o Qwen3.5 enxerga um exemplo de treino:\n")
print(sample[:1500])

In [ ]:
from datasets import Dataset

def build_ds(rows):
    ds = Dataset.from_list([{"text": to_training_text(e["messages"])} for e in rows])
    n0 = len(ds)
    ds = ds.filter(lambda x: len(tok(x["text"], add_special_tokens=False)["input_ids"]) <= MAX_SEQ_LEN)
    if len(ds) < n0:
        print(f"  {n0-len(ds)} exemplos acima de {MAX_SEQ_LEN} tokens removidos")
    return ds

train_ds, val_ds = build_ds(train), build_ds(val)
print(f"train={len(train_ds)} val={len(val_ds)}")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

CKPT_DIR = LOCAL_DIR / RUN_NAME / "checkpoints"
trainer = SFTTrainer(
    model=model, tokenizer=tok,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text", max_length=MAX_SEQ_LEN, dataset_num_proc=1,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS, learning_rate=LR, warmup_ratio=0.05,
        lr_scheduler_type="cosine", optim="adamw_8bit", weight_decay=0.001,
        logging_steps=5, eval_strategy="steps", eval_steps=50,
        save_strategy="steps", save_steps=50, save_total_limit=3,
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        seed=SEED, output_dir=str(CKPT_DIR), report_to="none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part=USER_TAG, response_part=ASSIST_TAG)

labels = trainer.train_dataset[0]["labels"]
print("Só isto entra na loss (deve ser apenas a resposta):\n")
print(tok.decode([t for t in labels if t != -100])[:600])

In [ ]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer_stats = trainer.train()
train_minutes = (time.time() - t0) / 60
peak_gb = torch.cuda.max_memory_reserved() / 1024**3

train_report = {
    "run": RUN_NAME, "load_in_4bit": LOAD_IN_4BIT, "lora_r": LORA_R, "epochs": EPOCHS, "lr": LR,
    "train_examples": len(train_ds), "minutes": round(train_minutes, 1), "peak_vram_gb": round(peak_gb, 2),
    "trainable_params_m": round(trainable / 1e6, 1), "gpu": torch.cuda.get_device_name(0),
    "best_eval_loss": trainer.state.best_metric,
}
(OUT_DIR / RUN_NAME).mkdir(parents=True, exist_ok=True)
json.dump(train_report, open(OUT_DIR / RUN_NAME / "train_report.json", "w"), indent=2)
json.dump(trainer.state.log_history, open(OUT_DIR / RUN_NAME / "log_history.json", "w"))
print(json.dumps(train_report, indent=2))

In [ ]:
import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(8, 4))
plt.plot(*zip(*tr), label="train loss", alpha=.7)
if ev:
    plt.plot(*zip(*ev), "o-", label="eval loss")
plt.xlabel("passo"); plt.ylabel("loss"); plt.title(f"Qwen3.5-2B · {RUN_NAME}"); plt.legend(); plt.grid(alpha=.3)
plt.savefig(OUT_DIR / RUN_NAME / "loss.png", dpi=160, bbox_inches="tight"); plt.show()

ADAPTER_DIR = OUT_DIR / RUN_NAME / "lora"
model.save_pretrained(str(ADAPTER_DIR)); tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Adapter salvo em", ADAPTER_DIR)

### RAG ponta a ponta no Colab
Mesma função `ask()` do notebook do Gemma. `use_adapter=False` usa o Qwen3.5-2B original (sem o LoRA) para comparação.

In [ ]:
Loader.for_inference(model)

def generate(messages, max_new_tokens=512, temperature=0.3):
    prompt = render(messages, generation=True)
    inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to("cuda")
    kw = dict(max_new_tokens=max_new_tokens, use_cache=True)
    kw.update(do_sample=True, temperature=temperature, top_p=0.95, top_k=20) if temperature > 0 else kw.update(do_sample=False)
    with torch.inference_mode():
        out = model.generate(**inputs, **kw)
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()

def ask(question, use_rag=True, use_adapter=True, k=TOP_K, temperature=0.3, show=True):
    ctx = hybrid_search(question, k=k) if use_rag else None
    msgs = build_messages(question, ctx)
    t0 = time.time()
    if use_adapter:
        answer = generate(msgs, temperature=temperature)
    else:
        with model.disable_adapter():
            answer = generate(msgs, temperature=temperature)
    dt = time.time() - t0
    if show:
        print(f"⏱ {dt:.1f}s\n{answer}")
        for i, c in enumerate(ctx or [], 1):
            print(f"   [{i}] {short_ref(c)} · {c['section']} · p.{c['page_start']}")
    return {"answer": answer, "contexts": ctx or [], "latency_s": dt}

q = "Por que a reintrodução de dispersores de sementes é importante para a regeneração da Mata Atlântica?"
for label, rag, ad in [("A · base", False, False), ("B · base + RAG", True, False),
                       ("C · fine-tunado", False, True), ("D · fine-tunado + RAG", True, True)]:
    print(f"\n{'='*20} {label} {'='*20}")
    ask(q, use_rag=rag, use_adapter=ad)

## 3. Avaliação — mesmas 4 configurações, mesmas perguntas, mesmo juiz
Como o split de teste é lido do mesmo arquivo e a amostragem usa a mesma semente, as perguntas são **as mesmas** usadas na avaliação do Gemma.

In [ ]:
import anthropic
llm = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

def call_llm(system, user, max_tokens=1500, retries=5):
    for attempt in range(retries):
        try:
            r = llm.messages.create(model=GEN_MODEL, max_tokens=max_tokens, system=system,
                                    messages=[{"role": "user", "content": user}])
            return "".join(b.text for b in r.content if b.type == "text")
        except Exception as e:
            wait = 2 ** attempt + random.random()
            print(f"  tentativa {attempt+1} falhou ({type(e).__name__}); aguardando {wait:.0f}s")
            time.sleep(wait)
    return None

def parse_json(txt):
    if not txt:
        return None
    txt = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
    a, b = txt.find("{"), txt.rfind("}")
    if a < 0 or b <= a:
        return None
    try:
        return json.loads(txt[a:b + 1])
    except json.JSONDecodeError:
        return None

ret_items = [e for e in test if e["kind"] in ("rag", "unanswerable")]
retrieval_report = retrieval_metrics(ret_items)
print("Recuperação (igual à do Gemma, mesmo índice):", retrieval_report)

In [ ]:
JUDGE_SYSTEM = "Você é um avaliador rigoroso de respostas sobre ecologia e conservação. Responda apenas com JSON."

def judge(question, reference, answer, contexts=None):
    ctx_block = f"\nCONTEXTO DISPONÍVEL AO MODELO:\n{format_context(contexts)}\n" if contexts else ""
    prompt = f"""PERGUNTA: {question}

RESPOSTA DE REFERÊNCIA: {reference}
{ctx_block}
RESPOSTA DO MODELO: {answer}

Avalie:
- "correcao" (1-5): a resposta do modelo transmite o mesmo conteúdo factual da referência? 5 = equivalente e completa; 3 = parcialmente correta; 1 = errada, irrelevante ou em outro idioma.
- "fidelidade" (1-5 ou null): {"tudo o que o modelo afirma está apoiado no CONTEXTO? 5 = nada inventado; 1 = muitas informações sem apoio." if contexts else "use null (não havia contexto)."}
Responda só com JSON: {{"correcao": int, "fidelidade": int|null, "comentario": "uma frase"}}"""
    return parse_json(call_llm(JUDGE_SYSTEM, prompt, max_tokens=300))

N_EVAL = 60
rng = random.Random(SEED)
test_rag = [e for e in test if e["kind"] == "rag"]
test_cb = [e for e in test if e["kind"] == "closed_book"]
test_un = [e for e in test if e["kind"] == "unanswerable"]
eval_items = rng.sample(test_rag, min(len(test_rag), int(N_EVAL * .7))) + rng.sample(test_cb, min(len(test_cb), int(N_EVAL * .3)))
eval_items = [{"question": e["question"], "reference": re.sub(r"\s?\[\d\]", "", e["reference"]),
               "kind": e["kind"], "doc_id": e["doc_id"]} for e in eval_items]
if GOLD_FILE.exists():
    eval_items += [{"question": g["question"], "reference": g["reference_answer"], "kind": "gold",
                    "doc_id": g.get("doc_id", "")} for g in read_jsonl(GOLD_FILE)]
print(f"Itens de avaliação: {len(eval_items)}", dict(Counter(e["kind"] for e in eval_items)))

CONFIGS = {"A: base": (False, False), "B: base+RAG": (True, False), "C: FT": (False, True), "D: FT+RAG": (True, True)}
rows = []
for i, e in enumerate(eval_items, 1):
    ctx = hybrid_search(e["question"])
    for name, (use_rag, use_ad) in CONFIGS.items():
        msgs = build_messages(e["question"], ctx if use_rag else None)
        t0 = time.time()
        if use_ad:
            ans = generate(msgs, temperature=0)
        else:
            with model.disable_adapter():
                ans = generate(msgs, temperature=0)
        lat = time.time() - t0
        j = judge(e["question"], e["reference"], ans, ctx if use_rag else None) or {}
        cites = [int(x) for x in re.findall(r"\[(\d)\]", ans)]
        cite_ok = None
        if use_rag and e["kind"] == "rag":
            cite_ok = bool(cites) and any(1 <= n <= len(ctx) and ctx[n - 1]["doc_id"] == e["doc_id"] for n in cites)
        rows.append({"config": name, "kind": e["kind"], "question": e["question"], "answer": ans,
                     "correcao": j.get("correcao"), "fidelidade": j.get("fidelidade"),
                     "cite_ok": cite_ok, "ptbr": is_ptbr(ans), "latency_s": lat})
    if i % 10 == 0:
        print(f"  {i}/{len(eval_items)}")

df_eval = pd.DataFrame(rows)
df_eval.to_csv(OUT_DIR / RUN_NAME / "eval_detalhado.csv", index=False)

In [ ]:
abst = []
for e in rng.sample(test_un, min(30, len(test_un))) + rng.sample(test_rag, min(30, len(test_rag))):
    ctx = [chunk_by_id[c] for c in e["context_chunk_ids"]]
    msgs = build_messages(e["question"], ctx)
    for name, use_ad in {"B: base+RAG": False, "D: FT+RAG": True}.items():
        if use_ad:
            ans = generate(msgs, temperature=0)
        else:
            with model.disable_adapter():
                ans = generate(msgs, temperature=0)
        abst.append({"config": name, "kind": e["kind"], "abstained": is_abstention(ans)})
df_abst = pd.DataFrame(abst)

summary = df_eval.groupby("config").agg(
    correcao=("correcao", "mean"), fidelidade=("fidelidade", "mean"),
    citacao_correta=("cite_ok", lambda s: s.dropna().astype(float).mean() if s.notna().any() else np.nan),
    ptbr=("ptbr", "mean"), latencia_s=("latency_s", "mean")).round(3)
if len(df_abst):
    summary["abstencao_correta"] = df_abst[df_abst.kind == "unanswerable"].groupby("config").abstained.mean()
    summary["falsa_abstencao"] = df_abst[df_abst.kind == "rag"].groupby("config").abstained.mean()
summary.to_csv(OUT_DIR / RUN_NAME / "eval_resumo.csv")
json.dump({"retrieval": retrieval_report, "summary": json.loads(summary.reset_index().to_json(orient="records"))},
          open(OUT_DIR / RUN_NAME / "eval_resumo.json", "w"), indent=2, ensure_ascii=False)
display(summary)

ax = summary[["correcao", "fidelidade"]].plot.bar(figsize=(8, 4), rot=0, ylim=(0, 5))
ax.set_title("Qwen3.5-2B · Refauna — notas do juiz (1–5)"); ax.grid(axis="y", alpha=.3)
plt.savefig(OUT_DIR / RUN_NAME / "eval_notas.png", dpi=160, bbox_inches="tight"); plt.show()

## 4. Gemma 4 E2B × Qwen3.5-2B
Junta os resultados salvos em `outputs/` pelos dois notebooks. **O slide principal da comparação sai daqui.**

> A `eval_loss` **não** é comparável entre os modelos (tokenizers diferentes contam tokens de forma diferente). Compare pelas notas do juiz, citação e abstenção.

In [ ]:
import matplotlib.pyplot as plt

rows = []
for rep in sorted(OUT_DIR.glob("*/train_report.json")):
    tr = json.load(open(rep))
    ev = rep.parent / "eval_resumo.csv"
    if not ev.exists():
        continue
    d = pd.read_csv(ev).set_index("config")
    for cfg in d.index:
        rows.append({"modelo": "Qwen3.5-2B" if tr["run"].startswith("qwen") else "Gemma 4 E2B",
                     "método": "QLoRA 4-bit" if tr["load_in_4bit"] else "LoRA 16-bit",
                     "run": tr["run"], "config": cfg, "min_treino": tr["minutes"], "vram_pico_gb": tr["peak_vram_gb"],
                     **d.loc[cfg].to_dict()})
cmp = pd.DataFrame(rows)
cmp.to_csv(OUT_DIR / "comparacao_modelos.csv", index=False)

metricas = [m for m in ["correcao", "fidelidade", "citacao_correta", "abstencao_correta", "falsa_abstencao", "ptbr", "latencia_s"] if m in cmp]
display(cmp.pivot_table(index="config", columns="run", values=metricas).round(2))
display(cmp.drop_duplicates("run")[["run", "modelo", "método", "min_treino", "vram_pico_gb"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, m in zip(axes, ["correcao", "fidelidade"]):
    cmp.pivot_table(index="config", columns="run", values=m).plot.bar(ax=ax, rot=0, ylim=(0, 5))
    ax.set_title(f"{m} (1–5)"); ax.grid(axis="y", alpha=.3); ax.legend(fontsize=8)
plt.suptitle("Refauna · Gemma 4 E2B × Qwen3.5-2B")
plt.savefig(OUT_DIR / "comparacao_modelos.png", dpi=160, bbox_inches="tight"); plt.show()

## 5. Publicação no Hugging Face
Três repositórios novos (o dataset já foi publicado pelo notebook do Gemma e é o mesmo). Tudo privado até você revisar.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
for repo in (REPO_LORA, REPO_MERGED, REPO_GGUF):
    api.create_repo(repo, private=HF_PRIVATE, exist_ok=True)

model.push_to_hub(REPO_LORA, token=HF_TOKEN)
tok.push_to_hub(REPO_LORA, token=HF_TOKEN)
print("✅ LoRA:", f"https://huggingface.co/{REPO_LORA}")

In [ ]:
model.push_to_hub_merged(REPO_MERGED, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
print("✅ Merged 16-bit:", f"https://huggingface.co/{REPO_MERGED}")

In [ ]:
model.push_to_hub_gguf(REPO_GGUF, tokenizer, quantization_method=["q4_k_m", "q8_0"], token=HF_TOKEN)
print("✅ GGUF:", f"https://huggingface.co/{REPO_GGUF}")

In [ ]:
def eval_table_md():
    try:
        s = json.load(open(OUT_DIR / RUN_NAME / "eval_resumo.json"))
    except FileNotFoundError:
        return "_Avaliação ainda não executada._"
    cols = ["config", "correcao", "fidelidade", "citacao_correta", "abstencao_correta", "ptbr"]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in s["summary"]:
        lines.append("| " + " | ".join("" if r.get(c) is None else (f"{r[c]:.2f}" if isinstance(r[c], float) else str(r[c])) for c in cols) + " |")
    rr = s["retrieval"]
    lines.append(f"\nRecuperação híbrida (teste, n={rr['n']}): Recall@{TOP_K} = {rr[f'recall@{TOP_K}']:.2f}, MRR@20 = {rr['mrr@20']:.2f}")
    return "\n".join(lines)

def model_card(library):
    refs = "\n".join(f"- {d['title']} ({d.get('year') or 's.d.'})" + (f". DOI: [{d['doi']}](https://doi.org/{d['doi']})" if d.get("doi") else "")
                     for d in sorted(docs, key=lambda d: (d.get("year") or 0)))
    return f"""---
license: apache-2.0
base_model: Qwen/Qwen3.5-2B
language: [pt, en]
library_name: {library}
pipeline_tag: text-generation
tags: [qwen3.5, unsloth, lora, rag, refauna, biodiversidade, conservacao, mata-atlantica, refaunacao]
---

# Qwen3.5-2B · Refauna (PT-BR)

Small Language Model especializado em **defaunação, refaunação, reintrodução de fauna, dispersão de sementes e conservação da Mata Atlântica**, a partir da produção científica do [Refauna](https://refauna.org.br).

Treinado com **os mesmos dados e o mesmo protocolo** do [Gemma 4 E2B · Refauna](https://huggingface.co/{HF_USERNAME}/gemma-4-e2b-refauna), para comparação entre modelos-base na palestra *"Pequenos modelos, conhecimento especializado: fine-tuning e RAG com os dados do Refauna"* (TDC Stage @ Rec'n'Play 2026).

## Como foi treinado
- **Base:** `Qwen/Qwen3.5-2B` (Apache 2.0) via `{BASE_MODEL}`
- **Método:** LoRA 16-bit (bf16), r={LORA_R}, alpha={LORA_R}, {EPOCHS} épocas, lr={LR}, atenção + MLP, template ChatML **sem raciocínio** (`enable_thinking=False`)
- **Dados:** {len(train)} exemplos sintéticos em PT-BR (estilo RAFT: com contexto + citações, abstenção e sem contexto), split por documento
- **Hardware:** {train_report['gpu']}, {train_report['minutes']} min, pico de {train_report['peak_vram_gb']} GB de VRAM

## Uso recomendado: com RAG e sem raciocínio
```
system: {SYSTEM_PROMPT}
user:   CONTEXTO:
        [1] (Autor et al., ano — Título) trecho...
        PERGUNTA: ...
```

### Ollama
```bash
ollama run hf.co/{REPO_GGUF}:Q4_K_M
```
Na API do Ollama, envie `"think": false`.

## Avaliação
{eval_table_md()}

A = base, B = base+RAG, C = fine-tunado, D = fine-tunado+RAG. Notas 1–5 de um juiz LLM.

## Limitações
- Uso educacional e de divulgação científica; **não substitui a literatura original**.
- Sem RAG, o modelo pode errar números, datas e nomes de espécies.

## Fontes (atribuição)
{refs}
"""

for repo, lib in [(REPO_LORA, "peft"), (REPO_MERGED, "transformers"), (REPO_GGUF, "gguf")]:
    api.upload_file(path_or_fileobj=model_card(lib).encode("utf-8"), path_in_repo="README.md",
                    repo_id=repo, commit_message="Model card")
print("✅ Model cards enviados")

## 6. No MacBook Air M1 (8 GB)

```bash
ollama pull qwen3.5:2b      # Qwen3.5-2B original, para a comparação ao vivo
ollama run hf.co/iamrodrigocoelho/qwen3.5-2b-refauna-GGUF:Q4_K_M
```

- **Raciocínio desligado:** no app local, envie `"think": false` nas chamadas à API do Ollama para os modelos Qwen. Se aparecer texto entre `<think>` e `</think>`, é sinal de que o raciocínio ficou ligado.
- Repositório privado: use a mesma chave SSH do Ollama que você cadastrou no Hugging Face para o Gemma.
- Memória: o GGUF Q4_K_M do Qwen3.5-2B tem tamanho parecido com o do Gemma 4 E2B; **rode um modelo por vez** no M1 de 8 GB (feche o outro com `ollama stop <modelo>`).
- **Teste de paridade:** rode as mesmas 10 perguntas aqui (`ask`) e no Ollama. Formato muito diferente = template de chat divergente no GGUF.